### **Telco Customer Churn Prediction - Feature Engineering**

**Objective:** To develop a machine learning classification model that predicts customers who are likely to discontinue their telecom subscription services. The project aims to identify the key factors influencing customer churn through data cleaning, feature engineering, model training, and evaluation, enabling telecom companies to implement targeted customer retention strategies and reduce revenue loss.

**Guiding Principles for this Notebook**

+ Minimal feature explosion → focus on quality over quantity
+ Preserve interpretability for business stakeholders
+ Transformations friendly to both tree-based models (Random Forest, XGBoost, LightGBM) and linear models

**Import Libraries**

In [1]:
import pandas as pd
import numpy as np

**Load and Inspect**

In [2]:
data = pd.read_csv("C:\\Users\\KOLADE\\OneDrive\\Documents\\AkoladeDSJourney\\Telco-Customer-Churn-Prediction\\data\\processed\\telco_customer_churn_cleaned.csv")
df = data.copy()

print(f"Data shape: {df.shape}") # check the shape of the dataset
df.head() # check the first 5 rows of the dataset

Data shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.rename(columns={'Churn': 'Churn_Status'}, inplace=True) # rename column for consistency
df['Churn'] = df['Churn_Status'].map({'Yes': 1, 'No': 0}) # create a new column for churn status

df.drop(columns=['Churn_Status', 'customerID'], inplace=True) # drop the original churn status and customerID columns
df.head() # check the first 5 rows of the dataset

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


**Feature Engineering**

In [4]:
 df['tenure'].describe()

count    7043.000000
mean       32.371149
std        24.559481
min         0.000000
25%         9.000000
50%        29.000000
75%        55.000000
max        72.000000
Name: tenure, dtype: float64

In [5]:
def tenure_category(tenure):
    if tenure <= 3:
        return '0-3 Months (Onboarding)'
    elif tenure <= 12:
        return '4-12 Months (Early Stage)'
    elif tenure <= 36:
        return '1-3 Years (Established)'
    else:
        return '3+ Months (Loyal)'

In [6]:
df['TenureGroup'] = df['tenure'].apply(tenure_category) # create a new column for tenure group
df[['tenure', 'TenureGroup']].head() # check the first 5 rows of the tenure and tenure group columns

,tenure,TenureGroup
0,1,0-3 Months (Onboarding)
1,34,1-3 Years (Established)
2,2,0-3 Months (Onboarding)
3,45,3+ Months (Loyal)
4,2,0-3 Months (Onboarding)


In [7]:
service = ['PhoneService', 'MultipleLines', 'InternetService',
            'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies']

df[service].head() # check the first 5 rows of the service columns 

,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies
0,No,No phone service,DSL,No,Yes,No,No,No,No
1,Yes,No,DSL,Yes,No,Yes,No,No,No
2,Yes,No,DSL,Yes,Yes,No,No,No,No
3,No,No phone service,DSL,Yes,No,Yes,Yes,No,No
4,Yes,No,Fiber optic,No,No,No,No,No,No


In [8]:
for col in service:
    print(f"{col} has {df[col].nunique()} unique values: {df[col].unique()}") # check the unique values of the service columns

PhoneService has 2 unique values: ['No' 'Yes']
MultipleLines has 3 unique values: ['No phone service' 'No' 'Yes']
InternetService has 3 unique values: ['DSL' 'Fiber optic' 'No']
OnlineSecurity has 3 unique values: ['No' 'Yes' 'No internet service']
OnlineBackup has 3 unique values: ['Yes' 'No' 'No internet service']
DeviceProtection has 3 unique values: ['No' 'Yes' 'No internet service']
TechSupport has 3 unique values: ['No' 'Yes' 'No internet service']
StreamingTV has 3 unique values: ['No' 'Yes' 'No internet service']
StreamingMovies has 3 unique values: ['No' 'Yes' 'No internet service']


In [9]:
df['NumServices'] = df[service].isin(['Yes', 'DSL', 'Fiber optic']).sum(axis=1) # create a new column for the number of services subscribed

df['NumServices'].value_counts() # check the value counts of the number of services subscribed

NumServices
1    1264
4     965
5     922
6     908
2     859
3     846
7     676
8     395
9     208
Name: count, dtype: int64

In [10]:
df[service + ['NumServices']].head() # check the first 5 rows of the service columns and the number of services subscribed

,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,NumServices
0,No,No phone service,DSL,No,Yes,No,No,No,No,2
1,Yes,No,DSL,Yes,No,Yes,No,No,No,4
2,Yes,No,DSL,Yes,Yes,No,No,No,No,4
3,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,4
4,Yes,No,Fiber optic,No,No,No,No,No,No,2


In [11]:
df['Contract'].value_counts() # check the value counts of the contract column

Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64

In [12]:
df['IsLongTermContract'] = df['Contract'].isin(['One year', 'Two year']).astype(int) # create a new column for long term contract

df[['Contract', 'IsLongTermContract']].head() # check the first 5 rows of the contract and long term contract columns

,Contract,IsLongTermContract
0,Month-to-month,0
1,One year,1
2,Month-to-month,0
3,One year,1
4,Month-to-month,0


In [13]:
df[['MonthlyCharges', 'TotalCharges', 'tenure']].describe() # check the descriptive statistics of the monthly charges, tenure and total charges columns

,MonthlyCharges,TotalCharges,tenure
count,7043.000000,7043.000000,7043.000000
mean,64.761692,2281.916928,32.371149
std,30.090047,2265.270398,24.559481
min,18.250000,18.800000,0.000000
25%,35.500000,402.225000,9.000000
50%,70.350000,1397.475000,29.000000
75%,89.850000,3786.600000,55.000000
max,118.750000,8684.800000,72.000000


In [14]:
df['AvgMonthlySpend'] = np.where(df['tenure'] > 0, round(df['TotalCharges'] / df['tenure'], 2), 0) # create a new column for average monthly spend

df[['MonthlyCharges', 'TotalCharges', 'tenure', 'AvgMonthlySpend']].head() # check the first 5 rows of the monthly charges, tenure, total charges and average monthly spend columns

,MonthlyCharges,TotalCharges,tenure,AvgMonthlySpend
0,29.85,29.85,1,29.85
1,56.95,1889.50,34,55.57
2,53.85,108.15,2,54.08
3,42.30,1840.75,45,40.91
4,70.70,151.65,2,75.82


In [15]:
df['SpendCategory'] = pd.qcut(
    df['AvgMonthlySpend'],
    q=4,
    labels=['Low', 'Medium', 'High', 'Premium']
) # create a new column for spend category based on AvgMonthlySpend charges

df[['AvgMonthlySpend', 'SpendCategory']].head() # check the first 5 rows of the average monthly spend and spend category columns

,AvgMonthlySpend,SpendCategory
0,29.85,Low
1,55.57,Medium
2,54.08,Medium
3,40.91,Medium
4,75.82,High


In [16]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TenureGroup,NumServices,IsLongTermContract,AvgMonthlySpend,SpendCategory
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,Yes,Electronic check,29.85,29.85,0,0-3 Months (Onboarding),2,0,29.85,Low
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,No,Mailed check,56.95,1889.50,0,1-3 Years (Established),4,1,55.57,Medium
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,Yes,Mailed check,53.85,108.15,1,0-3 Months (Onboarding),4,0,54.08,Medium
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,No,Bank transfer (automatic),42.30,1840.75,0,3+ Months (Loyal),4,1,40.91,Medium
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,Yes,Electronic check,70.70,151.65,1,0-3 Months (Onboarding),2,0,75.82,High


In [17]:
df.to_csv("C:\\Users\\KOLADE\\OneDrive\\Documents\\AkoladeDSJourney\\Telco-Customer-Churn-Prediction\\data\\processed\\telco_customer_churn_feature_engineered.csv", index=False) # save the feature engineered dataset to a new CSV file

print("Feature engineering completed and dataset saved to 'telco_customer_churn_feature_engineered.csv'")

Feature engineering completed and dataset saved to 'telco_customer_churn_feature_engineered.csv'


**Summary**

1. Create `TenureGroup` 
``` 
def tenure_category(tenure):
    if tenure <= 3:
        return '0-3 Months (Onboarding)'
    elif tenure <= 12:
        return '4-12 Months (Early Stage)'
    elif tenure <= 36:
        return '1-3 Years (Established)'
    else:
        return '3+ Months (Loyal)'
```
Captures Customer Lifecycle Stages

2. Create `NumServices` (Number of Services) - Manages Customer Engagement
3. Create `IsLongTermContract` - Strong Churn Predictor
4. Create `AvgMonthlySpend` - `TotalCharges` $\div$ `tenure`
5. Create `SpendCategory` - Distinguishes high value customers

**Next:** Telco Customer Churn Prediction - Hypothesis Testing

> + **notebook:** *`04_hypothesis_testing.ipynb`*
>
> + **create-src:** *`features.py`*